# Crop disease benchmark — revision experiments

**Before the first run** (right-hand panel → *Session options*):
1. **Accelerator:** GPU T4 ×1  **Internet:** On
2. **Add Input → Datasets:** search `20k multi-class crop disease images` (by *jawadali1045*)
   https://www.kaggle.com/datasets/jawadali1045/20k-multi-class-crop-disease-images
3. From session 2 onwards: **Add Input → Your Work → Notebooks → this notebook**, so the previous session's results are restored automatically.

**Every session:** set `SESSION` in the next cell, then **Save Version → Save & Run All (Commit)**. You can close the browser while it runs.

| SESSION | What runs | ≈ GPU time |
|---|---|---|
| 0 | Retrain the ORIGINAL single split (from GitHub) with MobileNetV3-Small + ConvNeXt-Tiny → accuracy on leaked vs clean test images (needs session 1 done) | ~1.5–2 h |
| 1 | Smoke test + data-integrity audit + crop-check images | ~1 h |
| 2 | 5 architectures, reference recipe, folds 0–1 | ~8.5 h |
| 3 | same, folds 2–3 | ~8.5 h |
| 4 | same, fold 4 + Grad-CAM / deletion AUC | ~4.5 h |
| 5 | Imbalance variants (cb, la, rs) on MobileNetV3-Small + ConvNeXt-Tiny, folds 0–1 | ~8.5 h |
| 6 | same, folds 2–3 | ~8.5 h |
| 7 | same, fold 4 | ~4.5 h |
| 8 | Learning-rate sensitivity (fold 0, 20 epochs) | ~9 h |
| 9 | Optional: imbalance variants ce + focal, all folds (run twice; finished runs are skipped) | ~10 h per session |
| 10 | Final analysis → all tables and figures (CPU is enough) | minutes |
| 11 | Imbalance variants (cb, la, rs) on MobileNetV3-Small + ConvNeXt-Tiny, fold 0 only | ~4 h |
| 12 | same, fold 1 only | ~4 h |
| 13 | same, fold 2 only | ~4 h |
| 14 | same, fold 3 only | ~4 h |
| 16 | Learning-rate sensitivity, part 1 (MobileNetV3-Small, ResNet-50, EfficientNetV2-S) | ~5 h |
| 17 | Learning-rate sensitivity, part 2 (ConvNeXt-Tiny, Swin-T) | ~4.5 h |
| 18 | Plain cross-entropy control (no label smoothing) on MobileNetV3-Small + ConvNeXt-Tiny, all 5 folds | ~7 h |

Sessions 11–14 together equal sessions 5–6, and 16–17 equal session 8: use whichever fits the remaining quota.

Finished runs are skipped automatically, so re-running a session after a crash only redoes the unfinished run.

In [ ]:
SESSION = 1          # <-- change this before every run

# Correct host-crop assignments here after checking the images in session 1 (leave {} if Wilt/Leaf Curl = cotton is right).
# Example: CROP_OVERRIDE = {"Army worm": "Maize"}
CROP_OVERRIDE = {}

## Setup (runs every session)

In [ ]:
import os, glob, json, shutil, subprocess, sys, time

os.environ["CDB_CROP_OVERRIDE"] = json.dumps(CROP_OVERRIDE)
try:
    print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
    HAS_GPU = True
except FileNotFoundError:
    HAS_GPU = False
    print("WARNING: no GPU attached (Session options -> Accelerator -> GPU T4 x1)")
if SESSION != 10 and not HAS_GPU:
    raise SystemExit("Training sessions need the GPU: switch the accelerator on and run again.")

REPO = "/kaggle/working/cdb"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/kapopat1980/crop-disease-benchmark.git", REPO], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagehash"], check=True)

# locate the dataset (the folder that contains Train/<class>/...)
hits = glob.glob("/kaggle/input/**/Tungro", recursive=True)
assert hits, "Dataset not found: add '20k Multi-Class Crop Disease Images' via Add Input"
DATA = os.path.dirname(os.path.dirname(hits[0]))
OUT = "/kaggle/working/revision_results"
print("DATA =", DATA)

## Write the two revision scripts (embedded copies; overwrite any older version from GitHub)

In [ ]:
%%writefile /kaggle/working/cdb/revision_experiments.py
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Revision experiments for the crop disease/pest benchmark (resubmission after The Visual Computer review).

Place this file next to crop_disease_benchmark.py (it re-uses its data scan, model factory,
transforms, metrics and Grad-CAM) and run the stages in order on Kaggle (Tesla T4).

Stages
------
  audit    (GPU optional, ~15 min)  cleaning audit per class and per reason, corrected label taxonomy,
                                    near-duplicate detection (pHash + ImageNet embeddings), duplicate
                                    groups, cross-split leakage check of the ORIGINAL seed-42 split,
                                    a montage of near-duplicate pairs for visual verification.
  cv       (GPU)                    5-fold stratified *group* cross-validation (near-duplicate groups never
                                    cross folds) on the corrected taxonomy, for any (model, variant).
                                    Saves val/test logits for every fold -> pooled out-of-fold evaluation.
  hpsweep  (GPU)                    learning-rate sensitivity per architecture on fold 0 (val macro-F1 only).
  explain  (GPU, ~10 min)           Grad-CAM for ALL models on the SAME fold-0 images: best cases (most
                                    confident correct) and worst cases (most confident errors), plus the
                                    deletion-AUC faithfulness score on a fixed random sample.

Training variants (--variants)
------------------------------
  ce_ls  cross-entropy + label smoothing 0.1          (= original recipe, the reference)
  ce     plain cross-entropy                          (isolates the effect of label smoothing on calibration)
  cb     class-balanced cross-entropy (effective number of samples, beta = 0.999) [Cui et al. 2019]
  focal  focal loss, gamma = 2 [Lin et al. 2017]
  la     logit-adjusted cross-entropy, tau = 1 [Menon et al. 2021]
  rs     square-root-frequency re-sampling of the training set (WeightedRandomSampler)

Suggested Kaggle commands (each block fits one 12-h session; runs are resumable)
------------------------------------------------------------------------------
  !python revision_experiments.py --stage audit
  !python revision_experiments.py --stage cv --models all --variants ce_ls                       # ~21 h, 2 sessions
  !python revision_experiments.py --stage cv --models MobileNetV3-Small ConvNeXt-Tiny \\
                                  --variants ce cb focal la rs                                    # ~35 h, 3-4 sessions
  !python revision_experiments.py --stage hpsweep --models all --epochs 20                         # ~9 h
  !python revision_experiments.py --stage explain --models all
Then, on any CPU:  python revision_analysis.py --out ./revision_results
"""
from __future__ import annotations

import argparse
import gc
import json
import math
import os
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, WeightedRandomSampler

sys.path.insert(0, str(Path(__file__).resolve().parent))
import crop_disease_benchmark as cdb  # noqa: E402

ALL_MODELS = cdb.ALL_MODELS
VARIANTS = ["ce_ls", "ce", "cb", "focal", "la", "rs"]

# -------------------------------------------------------------------------------------------------
# Corrected label taxonomy.
#   * merges: capitalisation / naming variants that denote the same disease
#   * crop: host crop for every class. Entries marked VERIFY were assigned from the dataset's folder
#     structure and confusion patterns; confirm them by inspecting the images before submission.
# -------------------------------------------------------------------------------------------------
MERGE = {
    "Wheat Brown leaf rust": "Wheat Brown leaf Rust",
    "bacterial_blight in Cotton": "Bacterial Blight in cotton",
}
RENAME = {  # cosmetic, for tables/figures only (does not change the task)
    "Wheat Brown leaf Rust": "Wheat brown (leaf) rust",
    "Bacterial Blight in cotton": "Cotton bacterial blight",
    "Becterial Blight in Rice": "Rice bacterial blight",
    "thirps on  cotton": "Cotton thrips",
    "Wheat___Yellow_Rust": "Wheat yellow rust",
    "Common_Rust": "Maize common rust",
    "Gray_Leaf_Spot": "Maize gray leaf spot",
    "Brownspot": "Rice brown spot",
    "Tungro": "Rice tungro",
    "Leaf smut": "Rice leaf smut",
    "Flag Smut": "Wheat flag smut",
    "Wilt": "Cotton wilt",
    "Leaf Curl": "Cotton leaf curl",
    "Army worm": "Armyworm",
}
CROP = {
    "Brownspot": "Rice", "Tungro": "Rice", "Leaf smut": "Rice", "Becterial Blight in Rice": "Rice", "Rice Blast": "Rice",
    "Common_Rust": "Maize", "Gray_Leaf_Spot": "Maize", "Healthy Maize": "Maize",
    "Flag Smut": "Wheat",
    "Wilt": "Cotton",        # VERIFY
    "Leaf Curl": "Cotton",   # VERIFY
    "Army worm": "Any",      # VERIFY: folder does not name a crop; "Any" = allowed under every crop mask
}
CROP.update(json.loads(os.environ.get("CDB_CROP_OVERRIDE", "{}")))  # optional override from the notebook


def crop_of(label: str) -> str:
    if label in CROP:
        return CROP[label]
    low = label.lower()
    for k in ["cotton", "wheat", "maize", "sugarcane", "rice"]:
        if k in low:
            return k.capitalize()
    return "Any"


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--stage", required=True, choices=["audit", "cv", "hpsweep", "explain", "orig"])
    p.add_argument("--data_path", default="/kaggle/input/datasets/jawadali1045/20k-multi-class-crop-disease-images")
    p.add_argument("--output_dir", default="./revision_results")
    p.add_argument("--orig_split", default="results/splits/split_seed42.csv",
                   help="split file of the original submission (for the leakage check)")
    p.add_argument("--models", nargs="+", default=["all"])
    p.add_argument("--variants", nargs="+", default=["ce_ls"])
    p.add_argument("--taxonomy", choices=["corrected", "original"], default="corrected")
    p.add_argument("--folds", type=int, default=5)
    p.add_argument("--fold_ids", nargs="+", type=int, default=None, help="subset of folds to run")
    p.add_argument("--cv_seed", type=int, default=42)
    p.add_argument("--phash_thr", type=int, default=6, help="max Hamming distance (64-bit pHash)")
    p.add_argument("--emb_thr", type=float, default=0.95, help="min cosine similarity of ImageNet embeddings")
    p.add_argument("--max_group_frac", type=float, default=0.01,
                   help="if a duplicate group exceeds this share of the data, group on pHash only")
    p.add_argument("--hp_lrs", nargs="+", type=float, default=[3e-5, 1e-4, 3e-4])
    # recipe (identical to the original submission unless overridden)
    p.add_argument("--img_size", type=int, default=288)
    p.add_argument("--batch_size", type=int, default=32)
    p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--patience", type=int, default=8)
    p.add_argument("--warmup_epochs", type=int, default=2)
    p.add_argument("--lr_backbone", type=float, default=1e-4)
    p.add_argument("--lr_head_mult", type=float, default=10.0)
    p.add_argument("--weight_decay", type=float, default=1e-2)
    p.add_argument("--head_dropout", type=float, default=0.3)
    p.add_argument("--grad_clip", type=float, default=1.0)
    p.add_argument("--num_workers", type=int, default=4)
    p.add_argument("--explain_n", type=int, default=6, help="best and worst cases per model")
    p.add_argument("--deletion_n", type=int, default=200)
    p.add_argument("--max_per_class", type=int, default=0, help="debug only")
    p.add_argument("--no_pretrained", action="store_true", help="debug only")
    p.add_argument("--force", action="store_true")
    a = p.parse_args()
    if a.models == ["all"]:
        a.models = list(ALL_MODELS)
    bad = set(a.variants) - set(VARIANTS)
    if bad:
        p.error(f"unknown variants {bad}")
    return a


# =================================================================================================
# STAGE 1: AUDIT
# =================================================================================================
def _scan_all(args):
    base = cdb.find_dataset(argparse.Namespace(data_path=args.data_path))
    mode, layout = cdb.discover_layout(Path(base))
    rows = []
    if mode == "predefined":
        for sp, root in layout.items():
            if root is not None:
                rows += cdb._scan(root, sp)
    else:
        rows += cdb._scan(layout, "all")
    return pd.DataFrame(rows, columns=["path", "label", "split"])


def cleaning_audit(args, out: Path) -> pd.DataFrame:
    """Re-runs the original cleaning but records the reason for every removed file."""
    df = _scan_all(args)
    res = [cdb._md5_and_check(p) for p in cdb.tqdm(df.path, desc="hash+verify")]
    df["md5"] = [h for h, _ in res]
    df["ok"] = [ok for _, ok in res]
    df["reason"] = np.where(df.ok, "kept", "corrupt")
    okm = df.ok
    nlab = df[okm].groupby("md5").label.nunique()
    conflict = set(nlab[nlab > 1].index)
    df.loc[okm & df.md5.isin(conflict), "reason"] = "label_conflict"
    live = df.reason == "kept"
    prio = {"test": 0, "val": 1, "train": 2}
    order = df[live].assign(_o=df.split.map(prio).fillna(3)).sort_values(["_o", "path"])
    dup = order.duplicated("md5", keep="first")
    df.loc[order.index[dup.values], "reason"] = "exact_duplicate_same_label"
    df.to_csv(out / "audit" / "cleaning_log.csv", index=False)
    per_class = df.pivot_table(index="label", columns="reason", values="path", aggfunc="count", fill_value=0)
    per_class["scanned"] = per_class.sum(1)
    per_class["removed_%"] = 100 * (1 - per_class.get("kept", 0) / per_class.scanned)
    per_class.sort_values("scanned", ascending=False).to_csv(out / "audit" / "cleaning_per_class.csv")
    summary = df.reason.value_counts().to_dict()
    summary["scanned"] = int(len(df))
    # where do conflicting labels come from? (pairs of labels sharing identical bytes)
    pairs = (df[okm & df.md5.isin(conflict)].groupby("md5").label.apply(lambda s: " | ".join(sorted(set(s))))
             .value_counts().rename_axis("label_set").reset_index(name="n_hashes"))
    pairs.to_csv(out / "audit" / "label_conflict_pairs.csv", index=False)
    cdb.save_json(summary, out / "audit" / "cleaning_summary.json")
    print("[audit] cleaning:", summary)
    return df[df.reason == "kept"].drop(columns=["ok", "reason"]).reset_index(drop=True)


def _phash_bits(paths):
    import imagehash
    H = np.zeros((len(paths), 64), dtype=np.uint8)
    for i, p in enumerate(cdb.tqdm(paths, desc="pHash")):
        with Image.open(p) as im:
            H[i] = imagehash.phash(im.convert("RGB")).hash.flatten().astype(np.uint8)
    return np.packbits(H, axis=1)  # (n, 8) uint8


def _hamming_pairs(packed, thr, chunk=512):
    n = len(packed)
    lut = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
    out = []
    for s in range(0, n, chunk):
        a = packed[s:s + chunk]
        d = lut[np.bitwise_xor(a[:, None, :], packed[None, :, :])].sum(-1)
        ii, jj = np.where(d <= thr)
        keep = (ii + s) < jj
        out += [(int(i + s), int(j), int(d[i, j])) for i, j in zip(ii[keep], jj[keep])]
    return out


@torch.inference_mode()
def _embeddings(paths, device, bs=64):
    from torchvision import models, transforms
    try:
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    except Exception as e:  # offline smoke test only
        print(f"[audit] WARNING: could not load ImageNet weights ({e}); embeddings are NOT meaningful")
        m = models.resnet50(weights=None)
    m.fc = nn.Identity()
    m = m.to(device).eval()
    tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                             transforms.Normalize(cdb.NORM_MEAN, cdb.NORM_STD)])
    ds = cdb.ImgDS(paths, [0] * len(paths), tf)
    E = []
    for x, _ in cdb.tqdm(DataLoader(ds, batch_size=bs, num_workers=2), desc="embed"):
        E.append(F.normalize(m(x.to(device)).float(), dim=1).cpu())
    return torch.cat(E)


def _cosine_pairs(E, thr, chunk=1024):
    out = []
    for s in range(0, len(E), chunk):
        sim = E[s:s + chunk] @ E.T
        ii, jj = torch.where(sim >= thr)
        keep = (ii + s) < jj
        out += [(int(i + s), int(j), float(sim[i, j])) for i, j in zip(ii[keep], jj[keep])]
    return out


def _union_find(n, edges):
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra
    return np.array([find(i) for i in range(n)])


def near_duplicate_audit(df, args, out: Path, device):
    paths = df.path.tolist()
    ph = _hamming_pairs(_phash_bits(paths), args.phash_thr)
    E = _embeddings(paths, device)
    cs = _cosine_pairs(E, args.emb_thr)
    pairs = {}
    for i, j, d in ph:
        pairs.setdefault((i, j), {})["hamming"] = d
    for i, j, s in cs:
        pairs.setdefault((i, j), {})["cosine"] = s
    rows = []
    for (i, j), v in pairs.items():
        rows.append(dict(i=i, j=j, hamming=v.get("hamming", np.nan), cosine=v.get("cosine", np.nan),
                         label_i=df.label[i], label_j=df.label[j], path_i=df.rel_path[i], path_j=df.rel_path[j]))
    P = pd.DataFrame(rows)
    group = _union_find(len(df), [(r.i, r.j) for r in P.itertuples()]) if len(P) else np.arange(len(df))
    biggest = np.bincount(group).max() if len(group) else 1
    if biggest > args.max_group_frac * len(df) and len(P):
        # transitive chaining of embedding neighbours can merge many merely similar leaves into one group;
        # fall back to pHash-only grouping (still reported as such in the summary)
        print(f"[audit] WARNING: largest group {biggest} images > {args.max_group_frac:.0%} of data; "
              "grouping on pHash pairs only (embedding pairs are still reported)")
        ph_edges = [(r.i, r.j) for r in P.itertuples() if not np.isnan(r.hamming)]
        group = _union_find(len(df), ph_edges)
        grouping_basis = "phash_only"
    else:
        grouping_basis = "phash_or_embedding"
    df = df.assign(group=group)
    sizes = df.group.map(df.group.value_counts())
    summ = {"pairs_total": int(len(P)),
            "pairs_phash": int(P.hamming.notna().sum()) if len(P) else 0,
            "pairs_embedding": int(P.cosine.notna().sum()) if len(P) else 0,
            "pairs_both": int((P.hamming.notna() & P.cosine.notna()).sum()) if len(P) else 0,
            "pairs_cross_label": int((P.label_i != P.label_j).sum()) if len(P) else 0,
            "images_in_groups_gt1": int((sizes > 1).sum()),
            "groups_gt1": int(df[sizes > 1].group.nunique()),
            "largest_group": int(sizes.max()), "grouping_basis": grouping_basis}
    # leakage in the ORIGINAL seed-42 split
    osp = Path(args.orig_split)
    if osp.exists() and len(P):
        o = pd.read_csv(osp)
        key = "rel_path" if "rel_path" in o.columns else "path"
        smap = dict(zip(o[key], o.split))
        P["split_i"] = P.path_i.map(smap) if key == "rel_path" else P.i.map(lambda k: smap.get(df.path[k]))
        P["split_j"] = P.path_j.map(smap) if key == "rel_path" else P.j.map(lambda k: smap.get(df.path[k]))
        cross = P[(P.split_i != P.split_j) & P.split_i.notna() & P.split_j.notna()]
        summ["orig_split_cross_pairs"] = int(len(cross))
        tst = set(cross[cross.split_i == "test"].path_i) | set(cross[cross.split_j == "test"].path_j)
        summ["orig_test_images_with_neardup_outside_test"] = int(len(tst))
        pd.Series(sorted(tst), name="rel_path").to_csv(out / "audit" / "orig_test_images_with_neardup.csv",
                                                        index=False)
    P.to_csv(out / "audit" / "near_duplicate_pairs.csv", index=False)
    per_class = df.assign(in_group=sizes > 1).groupby("label").in_group.agg(["sum", "count"])
    per_class.columns = ["images_in_dup_groups", "images"]
    per_class.to_csv(out / "audit" / "near_duplicates_per_class.csv")
    cdb.save_json(summ, out / "audit" / "near_duplicate_summary.json")
    print("[audit] near-duplicates:", summ)
    _montage(P, df, out / "audit" / "near_duplicate_examples")
    return df


def _montage(P, df, stem, n=12):
    import matplotlib.pyplot as plt
    if not len(P):
        return
    sel = pd.concat([P[P.label_i != P.label_j].head(n // 2), P[P.label_i == P.label_j].sample(
        min(n // 2, (P.label_i == P.label_j).sum()), random_state=0)])
    fig, ax = plt.subplots(len(sel), 2, figsize=(5, 2.4 * len(sel)))
    ax = np.atleast_2d(ax)
    for r, row in enumerate(sel.itertuples()):
        for c, (k, lab) in enumerate([(row.i, row.label_i), (row.j, row.label_j)]):
            with Image.open(df.path[k]) as im:
                ax[r, c].imshow(im.convert("RGB"))
            ax[r, c].set_title(f"{lab[:28]}", fontsize=8)
            ax[r, c].axis("off")
    fig.tight_layout()
    cdb.save_fig(fig, stem)
    plt.close(fig)


def stage_audit(args, out: Path, device):
    (out / "audit").mkdir(parents=True, exist_ok=True)
    kept = cleaning_audit(args, out)
    if args.max_per_class:
        kept = kept.groupby("label", group_keys=False).head(args.max_per_class).reset_index(drop=True)
    base = Path(cdb.find_dataset(argparse.Namespace(data_path=args.data_path)))
    kept["rel_path"] = [str(Path(p).relative_to(base)) if str(p).startswith(str(base)) else p for p in kept.path]
    kept = near_duplicate_audit(kept, args, out, device)
    kept["label_corrected"] = kept.label.map(lambda s: MERGE.get(s, s))
    kept["crop"] = kept.label.map(crop_of)
    kept.to_csv(out / "audit" / "master_index.csv", index=False)
    tax = (kept.groupby(["label", "label_corrected", "crop"]).size().rename("images").reset_index())
    tax["display_name"] = tax.label_corrected.map(lambda s: RENAME.get(s, s))
    tax.to_csv(out / "audit" / "taxonomy.csv", index=False)
    print(f"[audit] master index: {len(kept)} images | original classes {kept.label.nunique()} | "
          f"corrected classes {kept.label_corrected.nunique()}")


# =================================================================================================
# STAGE 2: GROUP-AWARE CROSS-VALIDATION
# =================================================================================================
def load_master(args, out: Path):
    m = pd.read_csv(out / "audit" / "master_index.csv")
    col = "label_corrected" if args.taxonomy == "corrected" else "label"
    classes = sorted(m[col].unique())
    m["y"] = m[col].map({c: i for i, c in enumerate(classes)})
    crops = [crop_of(c) for c in classes]
    return m, classes, crops


def fold_assignment(m, args):
    """Outer test fold via StratifiedGroupKFold; inner validation = one group-stratified sixth of the rest."""
    f = np.full(len(m), -1)
    sgk = StratifiedGroupKFold(n_splits=args.folds, shuffle=True, random_state=args.cv_seed)
    for k, (_, te) in enumerate(sgk.split(np.zeros(len(m)), m.y, m.group)):
        f[te] = k
    return f


def inner_val(m, train_idx, seed):
    sgk = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=seed)
    sub = m.iloc[train_idx]
    tr, va = next(sgk.split(np.zeros(len(sub)), sub.y, sub.group))
    return train_idx[tr], train_idx[va]


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.g = gamma

    def forward(self, logits, y):
        logp = F.log_softmax(logits.float(), 1).gather(1, y[:, None])[:, 0]
        return (-(1 - logp.exp()) ** self.g * logp).mean()


class LogitAdjustedLoss(nn.Module):
    def __init__(self, prior, tau=1.0):
        super().__init__()
        self.register_buffer("adj", tau * torch.log(prior.clamp_min(1e-12)))

    def forward(self, logits, y):
        return F.cross_entropy(logits.float() + self.adj, y)


def make_criterion(variant, ytr, C, device):
    counts = torch.tensor(np.bincount(ytr, minlength=C), dtype=torch.float32)
    if variant == "ce_ls":
        return nn.CrossEntropyLoss(label_smoothing=0.1)
    if variant in ("ce", "rs"):
        return nn.CrossEntropyLoss()
    if variant == "cb":
        return nn.CrossEntropyLoss(weight=cdb.effective_number_weights(ytr, C, 0.999).to(device))
    if variant == "focal":
        return FocalLoss(2.0)
    if variant == "la":
        return LogitAdjustedLoss((counts / counts.sum()).to(device))
    raise ValueError(variant)


@torch.inference_mode()
def predict_logits(model, loader, device):
    model.eval()
    L, Y = [], []
    for x, y in loader:
        with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            L.append(model(x.to(device, non_blocking=True)).float().cpu())
        Y.append(y)
    return torch.cat(L).numpy(), torch.cat(Y).numpy()


def macro_f1(y, p, C):
    from sklearn.metrics import f1_score
    return float(f1_score(y, p, labels=list(range(C)), average="macro", zero_division=0))


def train_eval(name, variant, m, tr, va, te, C, args, device, run_dir: Path, lr_backbone=None, save_test=True):
    res_path = run_dir / "result.json"
    if res_path.exists() and not args.force:
        print(f"  [skip] {run_dir}")
        return json.load(open(res_path))
    run_dir.mkdir(parents=True, exist_ok=True)
    seed = args.cv_seed
    cdb.set_seed(seed)
    tf_tr, tf_ev = cdb.get_transforms(args.img_size)
    ytr = m.y.values[tr]
    kw = dict(num_workers=args.num_workers, pin_memory=device.type == "cuda",
              worker_init_fn=cdb.seed_worker, persistent_workers=args.num_workers > 0)
    g = torch.Generator()
    g.manual_seed(seed)
    ds_tr = cdb.ImgDS(m.path.values[tr], ytr, tf_tr)
    if variant == "rs":
        cnt = np.bincount(ytr, minlength=C).astype(float)
        w = 1.0 / np.sqrt(cnt[ytr])
        sampler = WeightedRandomSampler(torch.tensor(w, dtype=torch.double), num_samples=len(tr),
                                        replacement=True, generator=g)
        train_l = DataLoader(ds_tr, batch_size=args.batch_size, sampler=sampler, drop_last=True, **kw)
    else:
        train_l = DataLoader(ds_tr, batch_size=args.batch_size, shuffle=True, drop_last=True, generator=g, **kw)
    val_l = DataLoader(cdb.ImgDS(m.path.values[va], m.y.values[va], tf_ev), batch_size=64, shuffle=False, **kw)

    model, head, _, _ = cdb.build_model(name, C, not args.no_pretrained, args.head_dropout)
    model = model.to(device)
    hid = {id(p) for p in head.parameters()}
    lrb = lr_backbone or args.lr_backbone
    opt = torch.optim.AdamW([{"params": [p for p in model.parameters() if id(p) not in hid], "lr": lrb},
                             {"params": list(head.parameters()), "lr": lrb * args.lr_head_mult}],
                            weight_decay=args.weight_decay)
    total, warm = args.epochs * len(train_l), max(1, args.warmup_epochs * len(train_l))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.01 + 0.99 * 0.5 * (1 + math.cos(
            math.pi * (s - warm) / max(1, total - warm))))
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    crit = make_criterion(variant, ytr, C, device)

    best, bad, hist, t0 = {"f1": -1, "loss": 1e9, "epoch": 0, "state": None}, 0, [], time.time()
    for ep in range(1, args.epochs + 1):
        model.train()
        tl, seen = 0.0, 0
        for x, y in train_l:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss = crit(model(x), y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            scaler.step(opt)
            scaler.update()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                sched.step()
            tl += loss.item() * y.size(0)
            seen += y.size(0)
        vl, vy = predict_logits(model, val_l, device)
        vf1 = macro_f1(vy, vl.argmax(1), C)
        vloss = float(F.cross_entropy(torch.tensor(vl), torch.tensor(vy)))
        hist.append(dict(epoch=ep, train_loss=tl / max(seen, 1), val_nll=vloss, val_f1=vf1))
        if vf1 > best["f1"] + 1e-6 or (abs(vf1 - best["f1"]) <= 1e-6 and vloss < best["loss"]):
            best.update(f1=vf1, loss=vloss, epoch=ep,
                        state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
            bad = 0
        else:
            bad += 1
        print(f"  [{name}/{variant}] ep {ep:02d} trL {hist[-1]['train_loss']:.4f} vaNLL {vloss:.4f} "
              f"vaF1 {vf1*100:.2f}{' *' if bad == 0 else ''}")
        if bad >= args.patience:
            break
    model.load_state_dict(best["state"])
    vl, vy = predict_logits(model, val_l, device)
    out = dict(model=name, variant=variant, lr_backbone=lrb, best_epoch=best["epoch"], epochs_run=len(hist),
               train_time_min=(time.time() - t0) / 60, n_train=len(tr), n_val=len(va), n_test=len(te),
               val_f1=macro_f1(vy, vl.argmax(1), C), val_acc=float((vl.argmax(1) == vy).mean()))
    pd.DataFrame(hist).to_csv(run_dir / "history.csv", index=False)
    arrays = dict(val_idx=va, val_logits=vl.astype(np.float16))
    if save_test:
        te_l = DataLoader(cdb.ImgDS(m.path.values[te], m.y.values[te], tf_ev), batch_size=64, shuffle=False, **kw)
        tl_, ty = predict_logits(model, te_l, device)
        arrays.update(test_idx=te, test_logits=tl_.astype(np.float16))
        out.update(test_f1=macro_f1(ty, tl_.argmax(1), C), test_acc=float((tl_.argmax(1) == ty).mean()))
        torch.save(best["state"], run_dir / "best.pt")
    np.savez_compressed(run_dir / "logits.npz", **arrays)
    cdb.save_json(out, res_path)
    print(f"  => {name}/{variant}: " + ", ".join(f"{k} {v:.4f}" for k, v in out.items() if k.startswith(("val_", "test_"))))
    del model, opt, best
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return out


def stage_cv(args, out: Path, device):
    m, classes, crops = load_master(args, out)
    C = len(classes)
    fold = fold_assignment(m, args)
    cvdir = out / f"cv_{args.taxonomy}"
    cvdir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"rel_path": m.rel_path, "y": m.y, "group": m.group, "fold": fold}).to_csv(
        cvdir / "folds.csv", index=False)
    cdb.save_json({"classes": classes, "crops": crops}, cvdir / "classes.json")
    for k in (args.fold_ids if args.fold_ids is not None else range(args.folds)):
        te = np.where(fold == k)[0]
        tr, va = inner_val(m, np.where(fold != k)[0], args.cv_seed + k)
        assert not (set(m.group[te]) & set(m.group[tr])), "group leakage"
        for name in args.models:
            for v in args.variants:
                print(f"\n=== fold {k} | {name} | {v} ===")
                train_eval(name, v, m, tr, va, te, C, args, device, cvdir / v / cdb.slug(name) / f"fold{k}")


def stage_orig(args, out: Path, device):
    """Retrain the ORIGINAL single split (results/splits/split_seed42.csv, 44 original classes, reference recipe)
    so that accuracy on test images WITH a near-duplicate outside the test set can be compared with the rest."""
    base = Path(cdb.find_dataset(argparse.Namespace(data_path=args.data_path)))
    sp = pd.read_csv(args.orig_split)
    m = pd.DataFrame({"path": [str(base / r) for r in sp.rel_path], "rel_path": sp.rel_path,
                      "y": sp.label_idx.astype(int), "split": sp.split})
    missing = [p for p in m.path if not Path(p).exists()]
    assert not missing, f"{len(missing)} split files not found under {base}, e.g. {missing[:2]}"
    C = int(m.y.max()) + 1
    tr, va, te = (np.where(m.split == s)[0] for s in ["train", "val", "test"])
    odir = out / "orig_split"
    odir.mkdir(parents=True, exist_ok=True)
    m[["rel_path", "y", "split"]].to_csv(odir / "split.csv", index=False)
    print(f"[orig] train/val/test = {len(tr)}/{len(va)}/{len(te)}, {C} classes")
    for name in args.models:
        train_eval(name, "ce_ls", m, tr, va, te, C, args, device, odir / cdb.slug(name) / "seed42")


def stage_hpsweep(args, out: Path, device):
    m, classes, _ = load_master(args, out)
    fold = fold_assignment(m, args)
    tr, va = inner_val(m, np.where(fold != 0)[0], args.cv_seed)
    for name in args.models:
        for lr in args.hp_lrs:
            train_eval(name, "ce_ls", m, tr, va, np.array([], int), len(classes), args, device,
                       out / "hpsweep" / cdb.slug(name) / f"lr{lr:g}", lr_backbone=lr, save_test=False)


# =================================================================================================
# STAGE 4: EXPLANATIONS (Grad-CAM best/worst, all models, same images) + deletion AUC
# =================================================================================================
def stage_explain(args, out: Path, device):
    import matplotlib.pyplot as plt
    m, classes, _ = load_master(args, out)
    C = len(classes)
    cvdir = out / f"cv_{args.taxonomy}"
    _, tf_ev = cdb.get_transforms(args.img_size)
    P = {}
    for name in args.models:
        z = np.load(cvdir / "ce_ls" / cdb.slug(name) / "fold0" / "logits.npz")
        P[name] = (z["test_idx"], torch.softmax(torch.tensor(z["test_logits"].astype(np.float32)), 1).numpy())
    idx = P[args.models[0]][0]
    y = m.y.values[idx]
    probs = np.stack([P[n][1] for n in args.models])            # (M, N, C)
    pt = probs[:, np.arange(len(idx)), y]                        # prob of true class
    allcorrect = (probs.argmax(2) == y).all(0)
    allwrong = (probs.argmax(2) != y).all(0)
    # consensus best/worst cases; relax to "majority of models" if too few unanimous cases exist
    maj_c = (probs.argmax(2) == y).mean(0) > 0.5
    best = [i for i in np.argsort(-pt.mean(0)) if allcorrect[i]] or [i for i in np.argsort(-pt.mean(0)) if maj_c[i]]
    worst = [i for i in np.argsort(pt.mean(0)) if allwrong[i]] or [i for i in np.argsort(pt.mean(0)) if not maj_c[i]]
    # one image per class where possible
    def distinct(lst, k):
        seen, sel = set(), []
        for i in lst:
            if y[i] not in seen:
                sel.append(i)
                seen.add(y[i])
            if len(sel) == k:
                break
        return sel
    sel = {"best": distinct(best, args.explain_n), "worst": distinct(worst, args.explain_n)}
    rng = np.random.default_rng(0)
    del_sample = rng.choice(len(idx), size=min(args.deletion_n, len(idx)), replace=False)
    rows = []
    for kind, ii in sel.items():
        if not ii:
            print(f"[explain] no {kind} cases found; skipped")
            continue
        fig, ax = plt.subplots(len(ii), len(args.models) + 1, figsize=(2.3 * (len(args.models) + 1), 2.5 * len(ii)))
        ax = np.atleast_2d(ax)
        for r, i in enumerate(ii):
            with Image.open(m.path.values[idx[i]]) as im:
                img = im.convert("RGB").resize((args.img_size, args.img_size))
            ax[r, 0].imshow(img)
            ax[r, 0].set_title(f"T: {classes[y[i]][:24]}", fontsize=8)
            ax[r, 0].axis("off")
        for c, name in enumerate(args.models, 1):
            model, _, layer, cl = cdb.build_model(name, C, False, args.head_dropout)
            model.load_state_dict(torch.load(cvdir / "ce_ls" / cdb.slug(name) / "fold0" / "best.pt",
                                             map_location="cpu"))
            model = model.to(device).eval()
            gc_ = cdb.GradCAM(model, layer, cl)
            for r, i in enumerate(ii):
                with Image.open(m.path.values[idx[i]]) as im:
                    x = tf_ev(im.convert("RGB"))[None].to(device)
                cam, logits = gc_(x)
                pr = torch.softmax(logits, 1)[0]
                ax[r, c].imshow(np.asarray(Image.open(m.path.values[idx[i]]).convert("RGB")
                                           .resize((args.img_size, args.img_size))))
                ax[r, c].imshow(cam[0], cmap="jet", alpha=0.45)
                ax[r, c].set_title(f"{name}\nP: {classes[int(pr.argmax())][:18]} ({float(pr.max()):.2f})", fontsize=7)
                ax[r, c].axis("off")
            gc_.remove()
            del model
            torch.cuda.empty_cache() if device.type == "cuda" else None
        fig.tight_layout()
        cdb.save_fig(fig, out / "figures" / f"gradcam_{kind}_all_models")
        plt.close(fig)
    for name in args.models:  # deletion AUC on the same random fold-0 sample for every model
        model, _, layer, cl = cdb.build_model(name, C, False, args.head_dropout)
        model.load_state_dict(torch.load(cvdir / "ce_ls" / cdb.slug(name) / "fold0" / "best.pt", map_location="cpu"))
        model = model.to(device).eval()
        gc_ = cdb.GradCAM(model, layer, cl)
        rows.append(dict(model=name, n_images=len(del_sample),
                         deletion_auc=_deletion_auc(model, gc_, m.path.values[idx[del_sample]], tf_ev, device)))
        gc_.remove()
        del model
    pd.DataFrame(rows).to_csv(out / "tables" / "deletion_auc.csv", index=False)
    print(pd.DataFrame(rows))


def _deletion_auc(model, gc_, paths, tf_ev, device, steps=10):
    """Mean area under the class-probability curve when the most salient pixels are removed first
    (lower = explanation is more faithful) [Petsiuk et al. 2018]. Removed pixels are set to 0 (= mean colour)."""
    xs_all, cams, cls = [], [], []
    for p in paths:
        with Image.open(p) as im:
            x = tf_ev(im.convert("RGB"))[None].to(device)
        cam, logits = gc_(x)
        xs_all.append(x)
        cams.append(cam[0])
        cls.append(int(logits.argmax()))
    gc_.remove()  # the Grad-CAM hook must not be active during the no-grad deletion passes
    aucs = []
    for x, cam, c in zip(xs_all, cams, cls):
        order = np.argsort(-cam.ravel())
        n = order.size
        xs = x.clone()
        probs = []
        with torch.inference_mode():
            for s in range(steps + 1):
                if s:
                    k = order[(s - 1) * n // steps: s * n // steps]
                    xs.view(1, 3, -1)[:, :, torch.as_tensor(k, device=device)] = 0
                probs.append(float(torch.softmax(model(xs), 1)[0, c]))
        aucs.append(np.trapezoid(probs, dx=1 / steps) if hasattr(np, "trapezoid") else np.trapz(probs, dx=1 / steps))
    return float(np.mean(aucs))


def main():
    args = parse_args()
    out = Path(args.output_dir)
    for d in ["audit", "tables", "figures"]:
        (out / d).mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.backends.cudnn.benchmark = True
    cdb.save_json(vars(args), out / f"config_{args.stage}.json")
    {"audit": stage_audit, "cv": stage_cv, "hpsweep": stage_hpsweep, "explain": stage_explain, "orig": stage_orig}[args.stage](
        args, out, device)
    print("Done.")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/cdb/revision_analysis.py
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
CPU-only analysis for the revised manuscript. Produces every table/figure value marked [[...]] in the
revised paper.

Two independent modes (use either or both):

  A) ORIGINAL seed-42 runs (available today, no training needed):
       python revision_analysis.py --orig_runs /path/to/journal_results \\
                                   --orig_split results/splits/split_seed42.csv \\
                                   [--neardup_list revision_results/audit/orig_test_images_with_neardup.csv]
     -> all 10 pairwise exact McNemar tests (Holm), signed calibration + reliability diagrams,
        per-class ROC diagnostics (MobileNetV3-Small AUC question), crop-conditioned inference,
        accuracy on test images with vs without a near-duplicate outside the test set.

  B) Revision cross-validation (after revision_experiments.py --stage cv):
       python revision_analysis.py --out revision_results [--taxonomy corrected]
     -> pooled out-of-fold metrics with bootstrap CIs and fold mean +/- SD, tail-class metrics,
        per-class recall with Wilson intervals, all-pairs McNemar + paired-bootstrap differences,
        imbalance-variant comparison, temperature scaling, crop-conditioned inference, hp sensitivity.
Outputs go to <out>/analysis/.
"""
from __future__ import annotations

import argparse
import json
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize_scalar
from sklearn.metrics import (balanced_accuracy_score, f1_score, matthews_corrcoef,
                             precision_recall_fscore_support, roc_auc_score)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

MODELS = ["MobileNetV3-Small", "ResNet-50", "EfficientNetV2-S", "ConvNeXt-Tiny", "Swin-T"]
VARIANTS = ["ce_ls", "ce", "cb", "focal", "la", "rs"]
VNAME = {"ce_ls": "CE + label smoothing (reference)", "ce": "CE", "cb": "Class-balanced CE",
         "focal": "Focal loss", "la": "Logit-adjusted CE", "rs": "Square-root re-sampling"}
TAIL_MAX = 100   # a class is a "tail" class if it has fewer than 100 images in total
plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11, "legend.fontsize": 10,
                     "savefig.dpi": 600})


def slug(s):
    return "".join(c if c.isalnum() else "_" for c in s).strip("_")


def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


# ------------------------------------------------------------------------------------------------- metrics
def core_metrics(y, P, C):
    p = P.argmax(1)
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, labels=list(range(C)), zero_division=0)
    present = np.bincount(y, minlength=C) > 0
    return {"accuracy": float((p == y).mean()),
            "macro_f1": float(f1[present].mean()),
            "balanced_accuracy": float(balanced_accuracy_score(y, p)),
            "mcc": float(matthews_corrcoef(y, p)),
            "nll": float(-np.log(np.clip(P[np.arange(len(y)), y], 1e-12, None)).mean()),
            "ece": ece(P, y), "signed_gap": signed_gap(P, y)}


def per_class(y, P, C):
    p = P.argmax(1)
    pr, rc, f1, sup = precision_recall_fscore_support(y, p, labels=list(range(C)), zero_division=0)
    return pd.DataFrame({"precision": pr, "recall": rc, "f1": f1, "support": sup})


def ece(P, y, bins=15):
    conf, acc = P.max(1), (P.argmax(1) == y).astype(float)
    e = 0.0
    for lo, hi in zip(np.linspace(0, 1, bins + 1)[:-1], np.linspace(0, 1, bins + 1)[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)


def signed_gap(P, y):
    """mean confidence - accuracy: > 0 over-confident, < 0 under-confident."""
    return float(P.max(1).mean() - (P.argmax(1) == y).mean())


def reliability(P, y, bins=15):
    conf, acc = P.max(1), (P.argmax(1) == y).astype(float)
    edges = np.linspace(0, 1, bins + 1)
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            rows.append(dict(lo=lo, hi=hi, n=int(m.sum()), conf=conf[m].mean(), acc=acc[m].mean()))
    return pd.DataFrame(rows)


def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    ph = k / n
    d = 1 + z * z / n
    c = (ph + z * z / (2 * n)) / d
    h = z * np.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / d
    return (max(0, c - h), min(1, c + h))


def mcnemar_exact(ca, cb):
    b = int((ca & ~cb).sum())
    c = int((~ca & cb).sum())
    p = 1.0 if b + c == 0 else float(stats.binomtest(min(b, c), b + c, 0.5).pvalue)
    return b, c, p


def holm(ps):
    ps = np.asarray(ps, float)
    o = np.argsort(ps)
    adj = np.empty_like(ps)
    run = 0.0
    for r, i in enumerate(o):
        run = max(run, (len(ps) - r) * ps[i])
        adj[i] = min(1.0, run)
    return adj


def boot_ci(y, P, C, fn, B=1000, seed=0):
    rng = np.random.default_rng(seed)
    v = []
    for _ in range(B):
        i = rng.integers(0, len(y), len(y))
        v.append(fn(y[i], P[i]))
    return np.percentile(v, [2.5, 97.5])


def paired_boot_diff(y, Pa, Pb, C, B=1000, seed=0):
    rng = np.random.default_rng(seed)
    f = lambda yy, PP: f1_score(yy, PP.argmax(1), labels=list(range(C)), average="macro", zero_division=0)
    d = []
    for _ in range(B):
        i = rng.integers(0, len(y), len(y))
        d.append(f(y[i], Pa[i]) - f(y[i], Pb[i]))
    return np.percentile(d, [2.5, 97.5])


def fit_temperature(logits, y):
    def nll(t):
        return -np.log(np.clip(softmax(logits / t)[np.arange(len(y)), y], 1e-12, None)).mean()
    return float(minimize_scalar(nll, bounds=(0.05, 10), method="bounded").x)


# ------------------------------------------------------------------------------------------------- crop
def crop_conditioned(P, y, crops):
    """Oracle crop known (deployment scenario): restrict the decision to classes of the image's crop.
    Two-stage: crop predicted by summing class probabilities per crop, then classify within it."""
    crops = np.asarray(crops)
    names = sorted(set(crops) - {"Any"})
    allow = {c: (crops == c) | (crops == "Any") for c in names}
    true_crop = crops[y]
    Po = P.copy()
    for i, c in enumerate(true_crop):
        if c != "Any":
            Po[i, ~allow[c]] = 0
    Po /= Po.sum(1, keepdims=True)
    crop_scores = np.stack([P[:, allow[c]].sum(1) for c in names], 1)
    pc = np.array(names)[crop_scores.argmax(1)]
    Pt = P.copy()
    for i, c in enumerate(pc):
        Pt[i, ~allow[c]] = 0
    Pt /= Pt.sum(1, keepdims=True)
    known = true_crop != "Any"
    crop_acc = float((pc[known] == true_crop[known]).mean())
    return Po, Pt, crop_acc


def error_breakdown(P, y, crops):
    crops = np.asarray(crops)
    p = P.argmax(1)
    wrong = p != y
    cross = wrong & (crops[p] != crops[y]) & (crops[p] != "Any") & (crops[y] != "Any")
    return int(wrong.sum()), int(cross.sum())


# ------------------------------------------------------------------------------------------------- ROC
def roc_diagnostics(y, P, C, sizes):
    rows = []
    for c in range(C):
        pos = y == c
        if 0 < pos.sum() < len(y):
            rows.append(dict(cls=c, n_pos=int(pos.sum()), n_total=int(sizes[c]),
                             auc=roc_auc_score(pos, P[:, c]),
                             recall=float((P.argmax(1)[pos] == c).mean())))
    R = pd.DataFrame(rows)
    p = P.argmax(1)
    wrong = p != y
    rank = (P > P[np.arange(len(y)), y][:, None]).sum(1)  # 0 = top-1
    return R, {
        "macro_auc": float(R.auc.mean()),
        "macro_auc_classes_ge10": float(R[R.n_pos >= 10].auc.mean()),
        "macro_auc_tail": float(R[R.n_total < TAIL_MAX].auc.mean()),
        "auc_recall_spearman": float(stats.spearmanr(R.auc, R.recall).correlation),
        "mean_max_prob": float(P.max(1).mean()),
        "mean_entropy": float((-(P * np.log(np.clip(P, 1e-12, None))).sum(1)).mean()),
        "errors_true_rank2_share": float((rank[wrong] == 1).mean()) if wrong.any() else np.nan,
        "errors_true_in_top3_share": float((rank[wrong] <= 2).mean()) if wrong.any() else np.nan,
        "mean_true_prob_on_errors": float(P[np.arange(len(y)), y][wrong].mean()) if wrong.any() else np.nan,
    }


# ------------------------------------------------------------------------------------------------- plots
def plot_reliability(curves, stem):
    n = len(curves)
    fig, ax = plt.subplots(1, n, figsize=(3.4 * n, 3.6), sharey=True)
    ax = np.atleast_1d(ax)
    for a, (name, items) in zip(ax, curves.items()):
        a.plot([0, 1], [0, 1], ls="--", c="grey", lw=1)
        for lab, R, st in items:
            a.plot(R.conf, R.acc, st, label=lab, ms=4)
        a.set_title(name)
        a.set_xlabel("Mean confidence")
        a.set_xlim(0, 1)
        a.set_ylim(0, 1)
        a.grid(alpha=.3)
    ax[0].set_ylabel("Accuracy")
    ax[0].legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    fig.savefig(str(stem) + ".png", bbox_inches="tight")
    fig.savefig(str(stem) + ".pdf", bbox_inches="tight")
    plt.close(fig)


def plot_roc_vs_recall(diag, stem):
    fig, ax = plt.subplots(figsize=(6, 4.2))
    for name, R in diag.items():
        ax.scatter(R.recall, R.auc, s=14 + 2 * np.sqrt(R.n_total), alpha=.6, label=name)
    ax.set_xlabel("Per-class recall (threshold-dependent)")
    ax.set_ylabel("Per-class ROC-AUC (ranking)")
    ax.grid(alpha=.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(str(stem) + ".png", bbox_inches="tight")
    fig.savefig(str(stem) + ".pdf", bbox_inches="tight")
    plt.close(fig)


# ================================================================================================= MODE A
def mode_original(args, out):
    split = pd.read_csv(args.orig_split)
    classes = sorted(split.label.unique())
    C = len(classes)
    sys_path = Path(__file__).resolve().parent
    import importlib.util
    spec = importlib.util.spec_from_file_location("rx", sys_path / "revision_experiments.py")
    crops = None
    try:
        rx = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(rx)
        crops = [rx.crop_of(c) for c in classes]
    except Exception as e:  # torch missing on this machine: fall back to a local copy of the rule
        print("[orig] crop map from fallback rule:", e)
    test = split[split.split == "test"].reset_index(drop=True)
    y = test.label_idx.values
    sizes = split.label_idx.value_counts().reindex(range(C), fill_value=0).values
    P = {}
    for name in MODELS:
        f = Path(args.orig_runs) / "runs" / slug(name) / "seed42" / "test_predictions.npz"
        if f.exists():
            z = np.load(f)
            assert (z["y_true"] == y).all(), f"{name}: test order differs from split file"
            P[name] = z["probs"].astype(np.float64)
            P[name] /= P[name].sum(1, keepdims=True)
    if not P:
        print("[orig] no test_predictions.npz found under", args.orig_runs)
        return
    o = out / "original_seed42"
    o.mkdir(parents=True, exist_ok=True)
    # 1. all pairwise McNemar
    rows = []
    for a, b in combinations(P, 2):
        ca, cb = P[a].argmax(1) == y, P[b].argmax(1) == y
        n10, n01, p = mcnemar_exact(ca, cb)
        lo, hi = paired_boot_diff(y, P[a], P[b], C)
        rows.append(dict(model_a=a, model_b=b, a_only=n10, b_only=n01, p_exact=p,
                         dF1_ci_lo=lo * 100, dF1_ci_hi=hi * 100))
    M = pd.DataFrame(rows)
    M["p_holm"] = holm(M.p_exact)
    M.to_csv(o / "mcnemar_all_pairs.csv", index=False)
    print(M.round(4).to_string())
    # 2. calibration (signed) + reliability
    cal, curves = [], {}
    for n, Pm in P.items():
        R = reliability(Pm, y)
        R.to_csv(o / f"reliability_{slug(n)}.csv", index=False)
        cal.append(dict(model=n, ece=ece(Pm, y) * 100, signed_gap=signed_gap(Pm, y) * 100,
                        mean_conf=Pm.max(1).mean() * 100, accuracy=(Pm.argmax(1) == y).mean() * 100,
                        share_samples_underconf_bins=float(R[R.acc > R.conf].n.sum() / R.n.sum())))
        curves[n] = [("seed 42 (label smoothing)", R, "o-")]
    pd.DataFrame(cal).to_csv(o / "calibration_signed.csv", index=False)
    print(pd.DataFrame(cal).round(3).to_string())
    plot_reliability(curves, o / "reliability_diagrams")
    # 3. ROC diagnostics
    diag, summ = {}, []
    for n, Pm in P.items():
        R, s = roc_diagnostics(y, Pm, C, sizes)
        R["class"] = [classes[c] for c in R.cls]
        R.to_csv(o / f"roc_per_class_{slug(n)}.csv", index=False)
        diag[n] = R
        summ.append(dict(model=n, **s))
    pd.DataFrame(summ).to_csv(o / "roc_diagnostics.csv", index=False)
    print(pd.DataFrame(summ).round(4).to_string())
    plot_roc_vs_recall(diag, o / "roc_vs_recall")
    # 4. crop-conditioned inference
    if crops is not None:
        rows = []
        for n, Pm in P.items():
            Po, Pt, cacc = crop_conditioned(Pm, y, crops)
            err, cross = error_breakdown(Pm, y, crops)
            for lab, PP in [("flat", Pm), ("crop known (oracle)", Po), ("two-stage (predicted crop)", Pt)]:
                mm = core_metrics(y, PP, C)
                rows.append(dict(model=n, inference=lab, accuracy=mm["accuracy"] * 100, macro_f1=mm["macro_f1"] * 100,
                                 errors=int((PP.argmax(1) != y).sum()), crop_accuracy=cacc * 100,
                                 cross_crop_errors_flat=cross))
        pd.DataFrame(rows).to_csv(o / "crop_conditioned.csv", index=False)
        print(pd.DataFrame(rows).round(2).to_string())
    # 5. near-duplicate effect
    if args.neardup_list and Path(args.neardup_list).exists():
        nd = set(pd.read_csv(args.neardup_list).rel_path)
        key = test.rel_path if "rel_path" in test.columns else test.path
        has = key.isin(nd).values
        rows = []
        for n, Pm in P.items():
            c = Pm.argmax(1) == y
            rows.append(dict(model=n, n_with=int(has.sum()), acc_with=c[has].mean() * 100 if has.any() else np.nan,
                             n_without=int((~has).sum()), acc_without=c[~has].mean() * 100))
        pd.DataFrame(rows).to_csv(o / "neardup_effect.csv", index=False)
        print(pd.DataFrame(rows).round(2).to_string())


# ================================================================================================= MODE B
def load_cv(cvdir: Path, variant, name, n_total):
    d = cvdir / variant / slug(name)
    folds = sorted(d.glob("fold*/logits.npz"))
    if not folds:
        return None
    test_idx, test_logits, fold_of, val = [], [], [], []
    for f in folds:
        z = np.load(f)
        k = int(f.parent.name[4:])
        test_idx.append(z["test_idx"])
        test_logits.append(z["test_logits"].astype(np.float64))
        fold_of.append(np.full(len(z["test_idx"]), k))
        val.append((z["val_idx"], z["val_logits"].astype(np.float64), k))
    return dict(idx=np.concatenate(test_idx), logits=np.concatenate(test_logits), fold=np.concatenate(fold_of),
                val=val, n_folds=len(folds))


def mode_cv(args, out):
    cvdir = out / f"cv_{args.taxonomy}"
    meta = json.load(open(cvdir / "classes.json"))
    classes, crops = meta["classes"], meta["crops"]
    C = len(classes)
    F = pd.read_csv(cvdir / "folds.csv")
    yall = F.y.values
    sizes = np.bincount(yall, minlength=C)
    tail = sizes < TAIL_MAX
    o = out / "analysis"
    o.mkdir(parents=True, exist_ok=True)
    runs = {}
    for v in VARIANTS:
        for n in MODELS:
            r = load_cv(cvdir, v, n, len(F))
            if r is not None:
                r["y"] = yall[r["idx"]]
                runs[(v, n)] = r
    complete = {k: r for k, r in runs.items() if r["n_folds"] == F.fold.nunique()}
    print(f"[cv] runs found: {len(runs)}; complete (all folds): {len(complete)}")

    rows, pcs = [], {}
    for (v, n), r in complete.items():
        y, Z = r["y"], r["logits"]
        P = softmax(Z)
        # temperature scaling, fitted per fold on that fold's validation logits
        Pts = np.zeros_like(P)
        temps = []
        for vidx, vlog, k in r["val"]:
            T = fit_temperature(vlog, yall[vidx])
            temps.append(T)
            m = r["fold"] == k
            Pts[m] = softmax(Z[m] / T)
        mm = core_metrics(y, P, C)
        fm = [core_metrics(y[r["fold"] == k], P[r["fold"] == k], C) for k in sorted(set(r["fold"]))]
        ci = boot_ci(y, P, C, lambda yy, PP: f1_score(yy, PP.argmax(1), labels=list(range(C)),
                                                         average="macro", zero_division=0))
        pc = per_class(y, P, C)
        pcs[(v, n)] = pc
        Po, Pt, cacc = crop_conditioned(P, y, crops)
        err, cross = error_breakdown(P, y, crops)
        _, rd = roc_diagnostics(y, P, C, sizes)
        rows.append(dict(
            variant=v, model=n,
            accuracy=mm["accuracy"] * 100, acc_fold_sd=np.std([f["accuracy"] for f in fm], ddof=1) * 100,
            macro_f1=mm["macro_f1"] * 100, macro_f1_ci_lo=ci[0] * 100, macro_f1_ci_hi=ci[1] * 100,
            macro_f1_fold_mean=np.mean([f["macro_f1"] for f in fm]) * 100,
            macro_f1_fold_sd=np.std([f["macro_f1"] for f in fm], ddof=1) * 100,
            balanced_accuracy=mm["balanced_accuracy"] * 100, mcc=mm["mcc"],
            tail_macro_f1=pc.f1[tail].mean() * 100, tail_macro_recall=pc.recall[tail].mean() * 100,
            head_macro_f1=pc.f1[~tail].mean() * 100,
            n_classes_recall0=int((pc.recall == 0).sum()),
            ece=mm["ece"] * 100, signed_gap=mm["signed_gap"] * 100, nll=mm["nll"],
            temperature=float(np.mean(temps)), ece_ts=ece(Pts, y) * 100, signed_gap_ts=signed_gap(Pts, y) * 100,
            nll_ts=core_metrics(y, Pts, C)["nll"],
            errors=err, cross_crop_errors=cross, crop_acc_two_stage=cacc * 100,
            acc_crop_known=core_metrics(y, Po, C)["accuracy"] * 100,
            macro_f1_crop_known=core_metrics(y, Po, C)["macro_f1"] * 100,
            macro_f1_two_stage=core_metrics(y, Pt, C)["macro_f1"] * 100,
            macro_auc=rd["macro_auc"] * 100, errors_true_in_top3=rd["errors_true_in_top3_share"] * 100))
    S = pd.DataFrame(rows)
    S.to_csv(o / "cv_summary.csv", index=False)
    if len(S):
        print(S.round(2).to_string())

    # per-class recall with Wilson CIs (reference variant + every variant of the two imbalance backbones)
    pr = []
    for (v, n), pc in pcs.items():
        for c in range(C):
            k = int(round(pc.recall[c] * pc.support[c]))
            lo, hi = wilson(k, int(pc.support[c]))
            pr.append(dict(variant=v, model=n, cls=classes[c], crop=crops[c], n=int(pc.support[c]),
                           precision=pc.precision[c], recall=pc.recall[c], f1=pc.f1[c], recall_lo=lo, recall_hi=hi))
    pd.DataFrame(pr).to_csv(o / "per_class_cv.csv", index=False)

    # macro-F1 deficit decomposition (which classes cost most), reference variant
    for (v, n), pc in pcs.items():
        if v == "ce_ls":
            pcs_def = (1 - pc.f1) / C * 100
            pcs_def.index = classes
            pcs_def.sort_values(ascending=False).rename("deficit_pp").to_csv(o / f"deficit_{slug(n)}.csv")

    # pairwise McNemar among architectures (reference variant) and variants vs reference within a model
    for label, pairs in [("architectures", [(("ce_ls", a), ("ce_ls", b)) for a, b in combinations(MODELS, 2)]),
                         ("variants", [((v, n), ("ce_ls", n)) for n in MODELS for v in VARIANTS if v != "ce_ls"])]:
        rows = []
        for ka, kb in pairs:
            if ka in complete and kb in complete:
                ra, rb = complete[ka], complete[kb]
                ia, ib = np.argsort(ra["idx"]), np.argsort(rb["idx"])
                assert (ra["idx"][ia] == rb["idx"][ib]).all()
                y = ra["y"][ia]
                Pa, Pb = softmax(ra["logits"][ia]), softmax(rb["logits"][ib])
                b, c, p = mcnemar_exact(Pa.argmax(1) == y, Pb.argmax(1) == y)
                lo, hi = paired_boot_diff(y, Pa, Pb, C)
                # tail-class recall difference (sign test on tail images)
                tm = tail[y]
                bt, ct, pt = mcnemar_exact((Pa.argmax(1) == y)[tm], (Pb.argmax(1) == y)[tm])
                rows.append(dict(a=f"{ka[1]} / {ka[0]}", b=f"{kb[1]} / {kb[0]}", a_only=b, b_only=c, p_exact=p,
                                 dF1_lo=lo * 100, dF1_hi=hi * 100, tail_a_only=bt, tail_b_only=ct, p_tail=pt))
        if rows:
            M = pd.DataFrame(rows)
            M["p_holm"] = holm(M.p_exact)
            M["p_tail_holm"] = holm(M.p_tail)
            M.to_csv(o / f"mcnemar_{label}.csv", index=False)
            print(M.round(4).to_string())

    # reliability diagrams (reference, CE, reference + temperature scaling)
    curves = {}
    for n in MODELS:
        items = []
        for v, st, ts in [("ce_ls", "o-", False), ("ce_ls", "s--", True), ("ce", "^-", False)]:
            r = complete.get((v, n))
            if r is None:
                continue
            Z, y = r["logits"], r["y"]
            if ts:
                P = np.zeros_like(Z)
                for vidx, vlog, k in r["val"]:
                    m = r["fold"] == k
                    P[m] = softmax(Z[m] / fit_temperature(vlog, yall[vidx]))
                lab = "CE+LS, temp.-scaled"
            else:
                P, lab = softmax(Z), VNAME[v].replace(" (reference)", "")
            items.append((lab, reliability(P, y), st))
        if items:
            curves[n] = items
    if curves:
        plot_reliability(curves, o / "reliability_cv")

    # hp sensitivity
    hp = []
    for f in sorted((out / "hpsweep").glob("*/lr*/result.json")):
        r = json.load(open(f))
        hp.append(dict(model=r["model"], lr_backbone=r["lr_backbone"], val_macro_f1=r["val_f1"] * 100,
                       val_acc=r["val_acc"] * 100, best_epoch=r["best_epoch"]))
    if hp:
        H = pd.DataFrame(hp).sort_values(["model", "lr_backbone"])
        H["lr_backbone"] = H.lr_backbone.map(lambda v: f"{v:g}")
        H.to_csv(o / "hp_sensitivity.csv", index=False)
        print(H.round(2).to_string())


def mode_orig_retrain(out):
    """Accuracy of models retrained on the original split, on test images with vs without a near-duplicate
    outside the test set (any detector, and hash-confirmed only)."""
    od = out / "orig_split"
    sp = pd.read_csv(od / "split.csv")
    pairs = pd.read_csv(out / "audit" / "near_duplicate_pairs.csv")
    def leaked(df):
        df = df[(df.split_i != df.split_j) & df.split_i.notna() & df.split_j.notna()]
        return set(df[df.split_i == "test"].path_i) | set(df[df.split_j == "test"].path_j)
    sets = {"any detector": leaked(pairs), "hash-confirmed": leaked(pairs[pairs.hamming.notna()])}
    rows = []
    for f in sorted(od.glob("*/seed42/logits.npz")):
        z = np.load(f)
        idx = z["test_idx"]
        y = sp.y.values[idx]
        P = softmax(z["test_logits"].astype(np.float64))
        ok = P.argmax(1) == y
        C = int(sp.y.max()) + 1
        base = dict(model=f.parent.parent.name, n_test=len(y), accuracy=ok.mean() * 100,
                    macro_f1=f1_score(y, P.argmax(1), labels=list(range(C)), average="macro", zero_division=0) * 100)
        for lab, S in sets.items():
            has = sp.rel_path.values[idx]
            has = np.array([h in S for h in has])
            k1, n1, k0, n0 = ok[has].sum(), has.sum(), ok[~has].sum(), (~has).sum()
            p = stats.fisher_exact([[k1, n1 - k1], [k0, n0 - k0]])[1] if n1 and n0 else np.nan
            lo1, hi1 = wilson(k1, n1)
            lo0, hi0 = wilson(k0, n0)
            rows.append(dict(base, leak_definition=lab, n_leaked=int(n1), acc_leaked=k1 / max(n1, 1) * 100,
                             acc_leaked_ci=f"[{lo1*100:.1f}, {hi1*100:.1f}]", n_clean=int(n0),
                             acc_clean=k0 / max(n0, 1) * 100, acc_clean_ci=f"[{lo0*100:.1f}, {hi0*100:.1f}]",
                             errors_leaked=int(n1 - k1), errors_clean=int(n0 - k0), p_fisher=p))
    if rows:
        R = pd.DataFrame(rows)
        (out / "analysis").mkdir(parents=True, exist_ok=True)
        R.to_csv(out / "analysis" / "orig_split_leakage_effect.csv", index=False)
        print(R.round(3).to_string())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="./revision_results")
    ap.add_argument("--taxonomy", default="corrected")
    ap.add_argument("--orig_runs", default=None)
    ap.add_argument("--orig_split", default="results/splits/split_seed42.csv")
    ap.add_argument("--neardup_list", default=None)
    a = ap.parse_args()
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    if a.orig_runs:
        mode_original(a, out / "analysis")
    if (out / "orig_split" / "split.csv").exists() and (out / "audit" / "near_duplicate_pairs.csv").exists():
        mode_orig_retrain(out)
    if (out / f"cv_{a.taxonomy}" / "folds.csv").exists():
        mode_cv(a, out)


if __name__ == "__main__":
    main()

## Restore results of earlier sessions

In [ ]:
prev = []
for depth in range(1, 6):   # Kaggle mounts notebook outputs at /kaggle/input/notebooks/<user>/<name>/...
    prev += glob.glob("/kaggle/input/" + "*/" * depth + "revision_results")
prev = [p for p in prev if os.path.isdir(os.path.join(p, "audit")) or glob.glob(p + "/*/*")]
if prev and not os.path.exists(OUT):
    src = max(prev, key=lambda p: len(glob.glob(p + "/**/result.json", recursive=True)))  # most complete copy
    print("restoring from", src)
    shutil.copytree(src, OUT)
elif not prev:
    print("No earlier results found. Mounted inputs:")
    for p in sorted(glob.glob("/kaggle/input/*/*/*"))[:20]:
        print("  ", p)
    if SESSION not in (1,):
        print("!! For sessions after 1: Add Input -> Your Work -> Notebooks -> this notebook, then run again.")
done = sorted(glob.glob(f"{OUT}/**/result.json", recursive=True))
print(f"{len(done)} finished runs found")
for d in done:
    print("  ", d.replace(OUT + "/", ""))

## Run this session

In [ ]:
def run(*args):
    cmd = [sys.executable, "-u", *args]
    print("\n$", " ".join(cmd), flush=True)
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    last = 0.0
    for line in p.stdout:
        if "it/s]" in line or "s/it]" in line:   # progress bar: show at most one line per minute
            if time.time() - last > 60:
                print("  ..", line.strip()[-110:], flush=True)
                last = time.time()
            continue
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed after {(time.time()-t0)/60:.1f} min")
    print(f"[ok, {(time.time()-t0)/60:.1f} min]")

X = ["revision_experiments.py", "--data_path", DATA, "--output_dir", OUT]
ALL = ["--models", "MobileNetV3-Small", "ResNet-50", "EfficientNetV2-S", "ConvNeXt-Tiny", "Swin-T"]
TWO = ["--models", "MobileNetV3-Small", "ConvNeXt-Tiny"]

if SESSION == 0:
    assert os.path.exists(f"{OUT}/audit/near_duplicate_pairs.csv"), "Run session 1 first and add this notebook as input"
    run(*X, "--stage", "orig", *TWO)
    run("revision_analysis.py", "--out", OUT)
elif SESSION == 1:
    smoke = "/kaggle/working/smoke"
    run("revision_experiments.py", "--stage", "audit", "--data_path", DATA, "--output_dir", smoke, "--max_per_class", "30")
    run("revision_experiments.py", "--stage", "cv", "--data_path", DATA, "--output_dir", smoke,
        "--models", "MobileNetV3-Small", "--fold_ids", "0", "--epochs", "1")
    shutil.rmtree(smoke)
    print("\n*** smoke test passed — running the real audit ***")
    run(*X, "--stage", "audit")
elif SESSION == 2:
    run(*X, "--stage", "cv", *ALL, "--variants", "ce_ls", "--fold_ids", "0", "1")
elif SESSION == 3:
    run(*X, "--stage", "cv", *ALL, "--variants", "ce_ls", "--fold_ids", "2", "3")
elif SESSION == 4:
    run(*X, "--stage", "cv", *ALL, "--variants", "ce_ls", "--fold_ids", "4")
    run(*X, "--stage", "explain", *ALL)
elif SESSION == 5:
    run(*X, "--stage", "cv", *TWO, "--variants", "cb", "la", "rs", "--fold_ids", "0", "1")
elif SESSION == 6:
    run(*X, "--stage", "cv", *TWO, "--variants", "cb", "la", "rs", "--fold_ids", "2", "3")
elif SESSION == 7:
    run(*X, "--stage", "cv", *TWO, "--variants", "cb", "la", "rs", "--fold_ids", "4")
elif SESSION == 8:
    run(*X, "--stage", "hpsweep", *ALL, "--epochs", "20")
elif SESSION == 9:
    # ~25 h in total: run SESSION 9 two or three times; Kaggle's 12-h limit may stop it, finished runs are kept
    run(*X, "--stage", "cv", *TWO, "--variants", "ce", "focal")
elif SESSION in (11, 12, 13, 14):
    run(*X, "--stage", "cv", *TWO, "--variants", "cb", "la", "rs", "--fold_ids", str(SESSION - 11))
elif SESSION == 16:
    run(*X, "--stage", "hpsweep", "--models", "MobileNetV3-Small", "ResNet-50", "EfficientNetV2-S", "--epochs", "20")
elif SESSION == 17:
    run(*X, "--stage", "hpsweep", "--models", "ConvNeXt-Tiny", "Swin-T", "--epochs", "20")
elif SESSION == 18:
    run(*X, "--stage", "cv", *TWO, "--variants", "ce")
elif SESSION == 10:
    run("revision_analysis.py", "--out", OUT)
else:
    raise ValueError("SESSION must be 0-14, 16, 17 or 18")

## Session 1 only: audit summary and images to check the host crop

In [ ]:
if SESSION == 1:
    import matplotlib.pyplot as plt
    from PIL import Image
    for f in ["cleaning_summary.json", "near_duplicate_summary.json"]:
        print(f, json.dumps(json.load(open(f"{OUT}/audit/{f}")), indent=1))
    ex = f"{OUT}/audit/near_duplicate_examples.png"
    if os.path.exists(ex):
        plt.figure(figsize=(8, 24)); plt.imshow(Image.open(ex)); plt.axis("off"); plt.title("Near-duplicate pairs: check they are real duplicates")
    for cls in ["Wilt", "Leaf Curl", "Army worm"]:
        fs = sorted(glob.glob(f"{DATA}/**/{cls}/*", recursive=True))[:8]
        fig, ax = plt.subplots(1, len(fs), figsize=(22, 3)); fig.suptitle(f"{cls}: which crop?", fontsize=14)
        for a, f in zip(ax, fs):
            a.imshow(Image.open(f).convert("RGB")); a.axis("off")
    plt.show()

## Progress report and a light backup zip (no model checkpoints) for download

In [ ]:
done = sorted(glob.glob(f"{OUT}/**/result.json", recursive=True))
print(f"{len(done)} finished runs in total")
rows = []
for d in done:
    r = json.load(open(d))
    rows.append((d.replace(OUT + "/", "").replace("/result.json", ""), r.get("test_f1", r.get("val_f1")), r.get("train_time_min")))
for k, f1, t in rows:
    print(f"  {k:55s} macro-F1 {f1*100:6.2f}   {t:5.1f} min")

if os.path.exists(OUT):
    light = "/kaggle/working/revision_results_light"
    shutil.rmtree(light, ignore_errors=True)
    shutil.copytree(OUT, light, ignore=shutil.ignore_patterns("*.pt"))
    shutil.make_archive("/kaggle/working/revision_results_light", "zip", light)
    shutil.rmtree(light)
    print("download from the Output tab: revision_results_light.zip")

## After SESSION 10
Download **revision_results_light.zip** from the *Output* tab and upload it to Claude; it contains `analysis/`, `audit/`,
`figures/` and `tables/` — everything needed to fill the manuscript placeholders.